### Prüfen NAs und 0 Values der Spalte 'rider_rank_season' 

In [1]:
import pandas as pd
import os


In [3]:
df = pd.read_pickle('../../data/processed/16_cleaned_master_data.pkl')

In [7]:

rank_nans = df['rider_rank_season'].isna().sum()
rank_zeros = (df['rider_rank_season'] == 0).sum()

print(f"In der Spalte 'rider_rank_season' gibt es:")
print(f"  - Fehlende Werte (NaN): {rank_nans} Zeilen")
print(f"  - Null-Werte (0):       {rank_zeros} Zeilen\n")

missing_rank_df = df[df['rider_rank_season'].isna()]
unique_affected_riders = missing_rank_df['rider_url'].nunique()
print(f"Davon betroffen sind: {unique_affected_riders} einzigartige Fahrer.")

In der Spalte 'rider_rank_season' gibt es:
  - Fehlende Werte (NaN): 1643 Zeilen
  - Null-Werte (0):       0 Zeilen

Davon betroffen sind: 36 einzigartige Fahrer.


Fahrer die nicht auftauchen also 0 Ranglistenpunkte -> 9999

In [8]:
missing_rank = df[df['rider_rank_season'].isna()]


recherche_rank_df = missing_rank[['name', 'year', 'nationality', 'rider_url']].drop_duplicates().copy()


recherche_rank_df['rider_rank_season_recherchiert'] = ""


recherche_rank_df = recherche_rank_df.sort_values(by=['name', 'year'])


target_path = "../../data/interim/36_fahrer_rank_recherche.csv"
recherche_rank_df.to_csv(target_path, index=False, sep=';')

In [10]:
recherche_rank = pd.read_csv('../../data/interim/36_fahrer_rank_recherche.csv', sep=';')


recherche_rank['rider_rank_season_recherchiert'] = pd.to_numeric(
    recherche_rank['rider_rank_season_recherchiert'], errors='coerce'
)


df = df.merge(
    recherche_rank[['rider_url', 'year', 'rider_rank_season_recherchiert']],
    on=['rider_url', 'year'],
    how='left'
)

df['rider_rank_season'] = df['rider_rank_season'].combine_first(df['rider_rank_season_recherchiert'])


df['rider_rank_season'] = df['rider_rank_season'].fillna(9999)

df.drop(columns=['rider_rank_season_recherchiert'], inplace=True)

df['rider_rank_season'] = df['rider_rank_season'].astype('Int64')

print(f"Verbleibende NaNs in 'rider_rank_season': {df['rider_rank_season'].isna().sum()}")
print(f"Datentyp der Ranking-Spalte:             {df['rider_rank_season'].dtype}")

Verbleibende NaNs in 'rider_rank_season': 0
Datentyp der Ranking-Spalte:             Int64


In [11]:
df.isna().sum()

race                             0
year                             0
url                              0
rank                             0
rider_url                        0
time_gap                         0
birthdate                        0
height                           0
name                             0
nationality                      0
weight                           0
url_name                         0
departure                        0
arrival                          0
distance                         0
vertical_meters                  0
won_how                          0
avg_speed                        0
race_ranking                     0
one_day_races                    0
gc                               0
time_trial                       0
sprint                           0
climber                          0
hills                            0
stage_nr                         0
date                             0
departure_lat                    0
departure_lon       

### Checkpoint
Neues .pkl erstellen 17

In [12]:
pfad = '../../data/processed/17_cleaned_master_data.pkl'
df.to_pickle(pfad)